In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split,RandomizedSearchCV,cross_val_score,KFold,StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder

Train_Data=pd.read_excel("C:/Users/Lenovo/OneDrive/Documents/Loan_Approval_Data.xlsx")
Train_Data["Loan_Status"] = Train_Data["Loan_Status"].map({
    "Approved": 1,
    "Rejected": 0
})
Train_Data=Train_Data.drop(columns=['Applicant_ID'])
scaler = StandardScaler()
Train_Data[['Age', 'Income', 'Credit_Score', 'Loan_Amount']] = scaler.fit_transform(
	Train_Data[['Age', 'Income', 'Credit_Score', 'Loan_Amount']]
)

encoder = LabelEncoder()
Train_Data['Employment_Type'] = encoder.fit_transform(Train_Data['Employment_Type'])
Train_Data['Existing_Loan'] = encoder.fit_transform(Train_Data['Existing_Loan'])
Train_Data['Loan_Status'] = encoder.fit_transform(Train_Data['Loan_Status'])

encoder2 = OneHotEncoder()
Train_Data[['Education_graduate','Education_highschool','Education_postgraduate']]=encoder2.fit_transform(Train_Data[['Education']]).toarray()

X=Train_Data.drop(columns=['Loan_Status'])
y=Train_Data['Loan_Status']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
model=XGBClassifier(
    n_estimators=15,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
param_dist = {
    'n_estimators':[5,10,15],
    'max_depth':[1,3,5]
}
random_search = RandomizedSearchCV(
    model,
    param_dist,
    n_iter=5,
    cv=5
)
X_train = X_train.drop(columns=['Education'])
X_test = X_test.drop(columns=['Education'])
random_search.fit(X_train, y_train)
y_pred=random_search.predict(X_test)
accuracy=accuracy_score(y_pred,y_test)
print(accuracy)
precision=precision_score(y_pred,y_test)
print(precision)
recall=recall_score(y_pred,y_test)
print(recall)
f1=f1_score(y_pred,y_test)
print(f1)
scores=cross_val_score(random_search,X.drop(columns='Education'),y,cv=5)

## K-Fold 
kfold=KFold(n_splits=5,shuffle=True,random_state=42)
scores_of_kfold=cross_val_score(random_search,X.drop(columns='Education'),y,cv=kfold,scoring='accuracy')
print(scores_of_kfold)
sfold=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores_of_sfold=cross_val_score(random_search,X.drop(columns='Education'),y,cv=sfold,scoring='accuracy')
print(scores_of_sfold)

print(scores)
print("Average Accuracy:", scores.mean())
print(Train_Data.corr(numeric_only=True))
Prediction_Data=pd.read_excel("C:/Users/Lenovo/OneDrive/Documents/Loan_Approval_Prediction_Data.xlsx")
Prediction_Data=Prediction_Data.drop(columns=['Applicant_ID'])
Prediction_Data[['Age', 'Income', 'Credit_Score', 'Loan_Amount']] = scaler.fit_transform(
	Prediction_Data[['Age', 'Income', 'Credit_Score', 'Loan_Amount']]
)

encoder = LabelEncoder()
Prediction_Data['Employment_Type'] = encoder.fit_transform(Prediction_Data['Employment_Type'])
Prediction_Data['Existing_Loan'] = encoder.fit_transform(Prediction_Data['Existing_Loan'])
##Train_Data['Loan_Status'] = encoder.fit_transform(Train_Data['Loan_Status'])

encoder2 = OneHotEncoder()
Prediction_Data[['Education_graduate','Education_highschool','Education_postgraduate']]=encoder2.fit_transform(Prediction_Data[['Education']]).toarray()
Prediction_Data=Prediction_Data.drop(columns=['Education'])
##print(Prediction_Data)
y_predictions=random_search.predict(Prediction_Data)
print("Loan_Status: ",y_predictions)







0.75
0.5
1.0
0.6666666666666666
[0.75 1.   1.   1.   1.  ]
[1.   1.   0.75 1.   1.  ]
[0.75 1.   1.   1.   1.  ]
Average Accuracy: 0.95
                             Age    Income  Credit_Score  Loan_Amount  \
Age                     1.000000  0.491309      0.295069     0.464397   
Income                  0.491309  1.000000      0.966095     0.923661   
Credit_Score            0.295069  0.966095      1.000000     0.866167   
Loan_Amount             0.464397  0.923661      0.866167     1.000000   
Employment_Type        -0.124545 -0.816099     -0.890484    -0.616141   
Existing_Loan          -0.124545 -0.816099     -0.890484    -0.616141   
Loan_Status             0.124545  0.816099      0.890484     0.616141   
Education_graduate     -0.715799 -0.416967     -0.211851    -0.365319   
Education_highschool    0.372891 -0.396073     -0.557093    -0.452182   
Education_postgraduate  0.502341  0.798354      0.699073     0.786190   

                        Employment_Type  Existing_Loan  Loan